# PyTorch Lightning fine-tuning template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a surya model to implement DS application.

It focuses on the concept of defining a modified Surya model, loading its weigths, and using a PyTorch lightning training loop to train it

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**

It doesn't require having seen the baselines template, but they are meant to complement each other.  **_In fact they are on purpose almost identical!!!_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# Lets the caching allocator grow a segment instead of reserving fixed-size blocks. This run
# allocates tensors of many different shapes -- every gradient-checkpoint recompute is a
# different shape -- which strands memory as "reserved but unallocated": visible in
# nvidia-smi, unusable by the next allocation. The failure this replaces reported 1.82 GiB
# stranded that way.
#
# torch 2.9 prints "PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead".
# Ignore it: PYTORCH_ALLOC_CONF is silently ignored for this setting on 2.9.1 (verified via
# torch.cuda.memory_snapshot()[i]["is_expandable"], which is only True under the old name).
# Do not "fix" the warning without re-checking that flag, or expandable segments turn off
# without a word. Unlike CUBLAS_WORKSPACE_CONFIG this only needs to precede the first CUDA
# *allocation*, not the torch import.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
from workshop_infrastructure.utils import cast_frozen_to, disable_peft_input_dtype_cast
torch.set_float32_matmul_precision('medium')



## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [3]:
# The config is the single source of truth. load_flare_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.sf_triggers.configs import load_flare_config

cfg = load_flare_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: solar_flare_forecasting


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [4]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


### Typed configuration, and how to extend it for your own task

`load_flare_config()` reads `configs/config_script.yaml` and returns a typed `TrainingConfig`.
This notebook and `3_finetune_template_1D.py` call the same function on the same file, so
there is no notebook-versus-script divergence to reason about.

| YAML section | Access in Python | Dataclass |
|---|---|---|
| `data:` | `cfg.data.*` | `FlareDataConfig` (this app) |
| `model:` | `cfg.model.*` | `ModelConfig` |
| `model.lora_config:` | `cfg.model.lora_config.*` | `LoraAdapterConfig` |
| `model.time_embedding:` | `cfg.model.time_embedding.*` | `TimeEmbeddingConfig` |
| `training:` | `cfg.learning_rate`, `cfg.batch_size`, … | `TrainingConfig` (flat) |
| `output:` | `cfg.output.*` | `OutputConfig` |
| `logging:` | `cfg.wandb_project`, `cfg.wandb_entity` | `TrainingConfig` (flat) |

**Everything except `FlareDataConfig` lives in `workshop_infrastructure/configs.py`** and is
shared by every downstream app. When you fork the template you do not copy that file. You
subclass `DataConfig` with your task's fields and bind `load_config` to it — the whole of
`downstream_apps/template/configs.py` is:

```python
@dataclass
class FlareDataConfig(DataConfig):
    flare_index_path: str = ""
    ds_time_column: str = "start_time"
    ds_time_tolerance: str = "4d"
    ds_match_direction: str = "forward"
    PATH_FIELDS = DataConfig.PATH_FIELDS + ("flare_index_path",)   # resolve it like a path

load_flare_config = partial(load_config, data_cls=FlareDataConfig)
```

Unknown keys are rejected rather than silently dropped: if you add a key to the YAML before
adding the field, you get an error naming the key and listing the valid ones.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additonal parameters relevant to the downstream application.  Here we focus in particular to the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is normalization.  Here we use a log normalization on xray flux that will act as the output target.  Making log10(xray_flux) strictly positive and having 66% of its values between 0 and 1

In this case we will define both a training and a validation dataset using the indices pointed at in the config

**_Important:  In this notebook we sets max_number_of_samples=6 to potentially avoid going through the whole dataset as we explore it.  Keep in mind this for the future in case the database seems smaller than you expect_**


In [5]:
from downstream_apps.sf_triggers.datasets.sf_triggers_dataset import FlareDSDataset

In [6]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the flare-specific arguments are passed here, which is exactly
# the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    FlareDSDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    # Host RAM, not GPU: this will not change torch.cuda.max_memory_allocated() at all.
    # One sample is (13, 1, 4096, 4096) fp32 = 832 MiB, and persistent_workers keeps the
    # validation workers resident through training, so the default prefetch_factor of 2 can
    # leave both loaders holding well over 10 GiB of pinned host memory. When that
    # overcommits, the workers die with "terminate called without an active exception" and
    # the traceback points nowhere near the cause.
    prefetch_factor=1,
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=10,
    ds_flare_index_path=cfg.data.flare_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
    ds_val_fraction=cfg.data.ds_val_fraction,
    ds_split_seed=cfg.data.ds_split_seed,
    mask_dir=cfg.data.mask_dir,
    mask_time_tolerance=cfg.data.mask_time_tolerance,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


train: 8 samples | val: 2 samples | batch_size: 1


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [7]:
# Inspect a single batch to confirm shapes before building the model.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})


{'ts': (1, 13, 1, 4096, 4096), 'time_delta_input': (1, 1), 'forecast': (1, 1, 4096, 4096), 'ds_index': 'list'}


## Initialize the HelioSpectformer model

This is the main difference beteween the notebook that trains the simple model and the one that fine-tunes Surya.  

In the case of the finetuning exercise one of the main differences between DS applications is the dimensionality of the output.  In this notebook we use a modified HelioSpectformer that projects into a 1D space. 

**_IMPORTANT: If your DS application is 2D you need to use the HelioSpectformer2D_**

In [8]:
from workshop_infrastructure.models.finetune_models import HelioSpectformer2D

/home/jovyan/envs/surya_ws/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Now the config file really comes into bear. The Spectformer has a metric ton of hyperparameters

In [9]:
# HelioSpectformer2D has a long list of architecture arguments, and all of them come
# straight from the model: section of the config. from_config() does that mapping, so the
# backbone can never drift out of sync with the checkpoint it is about to load.
#
# This app's target (a flare-region mask) is spatial, so it uses the 2D wrapper: its
# decoder head outputs (B, ft_out_chans, H, W), matching batch["forecast"]'s
# (B, 1, 4096, 4096) shape, unlike HelioSpectformer1D's pooled scalar output.
#
# Arguments that are not part of ModelConfig (dtype, and anything from the training:
# section) are passed as explicit overrides.
model = HelioSpectformer2D.from_config(
    cfg.model,
    ft_out_chans=1,
    dtype=cfg.dtype,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
)


## Load model weights

Here we load the pre-trained checkpoint and load the weights.  The exercise of loading follows the idea of us as many of the weights as possible.  This is accomplished through the filtered_checkpoint_state.   It checks to see if the pretrained model's layers match those of your finetuning architecture.   It also checks that all your dimensions across layers check out.   If something does not work those paramameters are left in their random initialization. 

In [10]:
# The checkpoint was saved from HelioSpectFormer directly, so its keys are flat
# (e.g. "embedding.proj.weight"), while the fine-tuning model nests the backbone under
# "backbone.*". load_pretrained_weights() tries both spellings and reports how many
# tensors matched — a low count means the architecture does not match the checkpoint.
from workshop_infrastructure.utils import load_pretrained_weights

load_pretrained_weights(model, cfg.model.pretrained_path)


Loading pretrained weights from /home/jovyan/adi_wdir/surya_workshop/downstream_apps/sf_triggers/assets/surya.366m.v1.pt.


Loaded 156 / 159 pretrained weights.


## To LoRA or not to Lora

This cell gives you two options.  On the one hand we have the classic freezing of the backbone (the initial layers of the model).   On the other hand we have the use of a LoRA.

LoRas have been a remarkable addition to our arsenal of models.   They have the advantage of keeping pretty much the entire model intact and only add broad modifications to weights as needed.

**What actually trains.** In the LoRA regime it is the adapters *and* the whole fine-tuning head. That second part is easy to get wrong: PEFT freezes every parameter it does not recognise as an adapter, so unless the head is explicitly re-enabled, the adapters end up fitting a **frozen, randomly initialised readout** — and the loss still goes down, so the training curve looks perfectly healthy. `apply_peft_lora()` avoids this by discovering every top-level module whose attribute name starts with `head_` and handing that list to PEFT as `modules_to_save`. The naming convention *is* the interface: a head component that does not carry the `head_` prefix is silently frozen, which is why `discover_head_modules()` validates it at startup and raises an error naming the attribute to rename.

**Where the adapters go.** `fc1`/`fc2` in all ten blocks, plus `attn.qkv` and `attn.proj` in the eight attention blocks. The spectral `complex_weight`, `attn.to_dynamic_projection`, and the patch embedding are never adapted.

Surya fuses query, key and value into a single `nn.Linear(1280, 3840)`, so one adapter covers all three at once: they share the `8×1280` matrix `A` and each gets its own `1280×8` slice of `B`. Their combined rank is at most 8 — which is *not* the same as giving q, k and v three independent rank-8 adapters.

Run the cell below and check the printout: the trainable-parameter count should be noticeably larger than the adapters alone (the exact number depends on the head architecture — here, `HelioSpectformer2D`'s spatial decoder).

### Two lines that decide whether this notebook fits in memory

The same cell applies two precision adjustments, and they are worth understanding rather than copying. Measured on this T4 at the full 4096×4096 config, the old settings hit `OutOfMemoryError` with 13.37 GiB allocated; with these two changes (plus `16-mixed` and one fix in the attention block) the same step peaks at **10.05 GiB and completes**, with 2.8 GiB of headroom.

**`disable_peft_input_dtype_cast(model)`.** PEFT's LoRA `Linear.forward` calls `_cast_input_dtype(x, lora_A.weight.dtype)` before the adapter branch. Under `*-mixed` precision the adapter weights are fp32 while `x` is half, so PEFT allocates a fresh fp32 copy of the activation — and autocast immediately casts it back down for `lora_A`. The copy accomplishes nothing, and it is billed at the widest tensor in the network. This is not a hypothetical: it is precisely the 1.25 GiB allocation that used to raise `OutOfMemoryError` in the `trainer.fit` cell at the bottom of this notebook.

**`cast_frozen_to(model, torch.float16)`.** Only 0.2% of this model trains. The other 99.8% is frozen, never sees an optimizer update, and has no reason to sit in fp32 while autocast casts it down on every single matmul anyway. Casting it once up front halves it. The lever most people miss is that this must cover **buffers** too, not just parameters — see the comment in the cell for why one 320 MiB buffer was quietly costing 1.7 GiB.

Neither of these is exotic; both are consequences of one idea worth carrying to your own fine-tuning work: *under mixed precision, the only tensors that need fp32 are the ones the optimizer updates.*

In [11]:
# Three fine-tuning regimes, all selected from the model: section of the config:
#
#   use_lora: true                          -> LoRA adapters + the whole head (default)
#   use_lora: false, freeze_backbone: true  -> linear probe: only the head trains
#   use_lora: false, freeze_backbone: false -> full fine-tuning of all 366M parameters
#
# freeze_backbone is ignored when use_lora is true: PEFT freezes everything, then
# re-enables the adapters and every head_* module.
#
# 3_finetune_template_1D.py applies exactly this logic in build_model().
if cfg.model.freeze_backbone:
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = False

if cfg.model.use_lora:
    # Prints the adapted modules and the trainable head modules it discovered.
    model = apply_peft_lora(model, cfg.model.lora_config)

    # Memory lever 1. PEFT aligns each adapter's input with the adapter weight dtype, which
    # under mixed precision means upcasting a half activation to fp32 -- only for autocast to
    # cast it straight back down for the very next matmul. The copy is pure waste, and it is
    # charged at the widest point in the network: mlp.fc2's input here is (1, 65536, 5120),
    # so the copy is exactly 1.25 GiB, per block, on every gradient-checkpoint recompute.
    # That specific allocation is what used to OOM this notebook.
    disable_peft_input_dtype_cast(model)

# Memory lever 2. Frozen weights and buffers to half; anything trainable stays fp32 so the
# optimizer keeps a full-precision master copy. This has to come after the weight load
# (so the checkpoint is read at full precision and rounded exactly once) and after the
# requires_grad decisions above (so "frozen" means what we just decided it means).
#
# The non-obvious win is a *buffer*, not a weight: LinearEmbedding.pos_embed is a
# (1, 65536, 1280) fp32 buffer that gets added to the half-precision patch-embedding output.
# fp32 + half promotes back to fp32, so leaving it alone pins the entire inter-block
# residual stream to fp32 -- with all ten layers checkpointed that is 11 live token tensors,
# 3.44 GiB instead of 1.72 GiB.
#
# Normalization layers are held back at fp32 on purpose; see the helper's docstring.
# Use torch.bfloat16 instead on SM80+ (A100/H100).
if cfg.precision.endswith("-mixed"):
    cast_frozen_to(model, torch.float16)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Applying PEFT LoRA: r=2, alpha=8, dropout=0.0, modules=['fc1', 'fc2', 'attn.qkv', 'attn.proj']


[LoRA] Adapted modules (36):
[LoRA]   backbone.backbone.blocks_attention.0.attn.proj
[LoRA]   backbone.backbone.blocks_attention.0.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.1.attn.proj
[LoRA]   backbone.backbone.blocks_attention.1.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.2.attn.proj
[LoRA]   backbone.backbone.blocks_attention.2.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.3.attn.proj
[LoRA]   backbone.backbone.blocks_attention.3.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.4.attn.proj
[LoRA]   backbone.backbone.blocks_atten

[PRECISION] Cast 86 frozen parameter(s) and 1 buffer(s) to torch.float16; left 36 normalization layer(s) in fp32.
Trainable parameters: 706,816 / 358,701,824 (0.20%)


We can now test that this model manipulates a batch as expected and returns an estimate of flare intensity as we did for the simple baseline.

We pass the input stack 'ts' to the model to transform it into our regression output.   Note that since this model was trained for a different task, it's likely it won't perform very well.  As with the simple baseline, this only acts as a test that our model forward doesn't have dimension problems.

Dimension problemns are the dominant source of error in this kind of work.

Note that our output has now the size of our batch.

**One consequence of the cell above:** the frozen part of the model is now fp16, so every ad-hoc forward pass has to run inside `torch.autocast`. A bare `model.forward(batch)` raises `expected scalar type Half but found Float` from the patch-embedding `Conv2d`. Lightning does this for you during `fit()`; you only have to think about it when calling the model by hand, as we do here.

In [25]:
batch = next(iter(train_data_loader))

# The autocast wrapper is required, not cosmetic. The cell above cast the frozen weights to
# fp16, so a bare model.forward(batch) now raises "expected scalar type Half but found
# Float" from the patch-embedding Conv2d -- autocast is what reconciles the half backbone
# with the fp32 adapters and head. This still runs on the CPU: nothing has been moved to the
# GPU yet, which is deliberate, because a shape bug is much cheaper to find here.
#
# Note there is no torch.no_grad() here on purpose: the metric cells below reuse `output`
# and expect it to still carry its autograd graph.
device_type = "cuda" if torch.cuda.is_available() else "cpu"
with torch.autocast(device_type, dtype=torch.float16):
    output = model.forward(batch)

output

RuntimeError: Input type (float) and bias type (c10::Half) should be the same

### Aside: measure the memory, don't guess it

Lightning will print `1,434.807 MB total estimated model params size` when training starts, and it is easy to read that as "the model needs 1.4 GB". It is the *static* cost only. What actually decides whether this run fits is the **transient** peak, and for this architecture the peak lives somewhere nothing in the training log looks: inside a gradient-checkpoint recomputation during the backward pass.

The arithmetic is worth doing once by hand. At `img_size 4096` and `patch_size 16` there are `(4096/16)² = 65,536` tokens, `embed_dim` is 1280, and `mlp_ratio 4` makes the MLP hidden dimension 5120. So one MLP hidden activation is

```
1 × 65,536 × 5,120 × 4 bytes (fp32) = 1.25 GiB
```

for a **single sample**. Every block that gets recomputed in backward allocates that again. `checkpoint_layers: [0..9]` trades exactly this compute-for-memory: block inputs are kept, everything inside is thrown away and rebuilt. That is why the peak is invisible in the forward pass and why "reduce the batch size" stops helping once you are already at 1.

The cell below runs one real forward+backward and reports the peak. Run it once per change you make, and write the number down — that is how you find out which lever actually mattered instead of changing five things and hoping.

In [13]:
import gc

import torch

# Match training.precision. On this T4 (SM75) that is fp16: native tensor cores, whereas
# bf16 is emulated and slower than fp32. Use torch.bfloat16 on SM80+.
PROBE_PRECISION = torch.float16


def probe_step_memory(model, batch, autocast_dtype=PROBE_PRECISION, label=""):
    """Run one forward+backward on the GPU and report peak memory, then clean up fully."""
    device = torch.device("cuda")
    model = model.to(device)
    gpu_batch = {
        k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    static = torch.cuda.memory_allocated()

    # autocast is not optional here. The frozen backbone is half precision and the adapters
    # are fp32; autocast is what reconciles the two. Without it the patch-embedding Conv2d
    # raises "expected scalar type Half but found Float".
    with torch.autocast("cuda", dtype=autocast_dtype):
        out = model(gpu_batch)
        loss = torch.nn.functional.mse_loss(
            out.reshape(-1), gpu_batch["forecast"].reshape(-1).float()
        )
    loss.backward()

    peak_alloc = torch.cuda.max_memory_allocated() / 2**30
    peak_resv = torch.cuda.max_memory_reserved() / 2**30
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"{label or 'probe'}:")
    print(f"  static (weights on GPU)         : {static / 2**30:6.2f} GiB")
    print(f"  peak allocated                  : {peak_alloc:6.2f} GiB")
    print(f"  peak reserved                   : {peak_resv:6.2f} GiB")
    print(f"  reserved but unallocated (frag)  : {peak_resv - peak_alloc:6.2f} GiB")
    print(f"  device total                    : {total:6.2f} GiB "
          f"-> headroom {total - peak_resv:5.2f} GiB")
    print(f"  loss                            : {loss.item():.6f}  (output {out.dtype})")

    # Give every byte back, so trainer.fit() below starts from a clean pool. Leaving the
    # weights resident and letting Lightning move them again is a reliable way to OOM the
    # first real training step for reasons unrelated to the model.
    del out, loss, gpu_batch
    model.zero_grad(set_to_none=True)
    model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    return peak_alloc


if torch.cuda.is_available():
    _ = probe_step_memory(model, batch, label="current settings")
else:
    print("No GPU visible; skipping the memory probe.")

current settings:
  static (weights on GPU)         :   1.70 GiB
  peak allocated                  :  10.10 GiB
  peak reserved                   :  11.93 GiB
  reserved but unallocated (frag)  :   1.82 GiB
  device total                    :  14.74 GiB -> headroom  2.82 GiB


  loss                            : 0.223045  (output torch.float16)


## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantitification of error, which in turn shifts the weights towards better pefrorming models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics do you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can be later use in a training script

In [14]:
from downstream_apps.sf_triggers.metrics.sf_triggers_metrics import FlareMetrics

In [15]:
train_loss_metrics = FlareMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same MSE as train_loss — override FlareMetrics.val_loss to change it.
val_loss_metrics = FlareMetrics("val_loss")
train_evaluation_metrics = FlareMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = FlareMetrics("val_metrics")

Now they can be evaluated in our model's output and our ground truth.   First the loss that actually will backpropagate, in this case Mean Squared Errror

In [16]:
train_loss_metrics(output, batch["forecast"])

NameError: name 'output' is not defined

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it her as an example, but oftentimes is better to put the diagnostics only in the validation evaluation metrics.

Here we are caclulating the Root Relative Squared Error https://lightning.ai/docs/torchmetrics/stable/regression/rse.html 

A value below one means the prediction is better than predicting the average.  It is unlikely that this metric will be lower than one with a randomly initialized model

In [17]:
train_evaluation_metrics(output, batch["forecast"])

NameError: name 'output' is not defined

In the validation evaluation metrics we report both MSE and RRSE

In [18]:
validation_evaluation_metrics(output, batch["forecast"])

NameError: name 'output' is not defined

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the FlareLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

**_Note that it is the same Lightning module we used for the baseline!!_**

In [19]:
from downstream_apps.template.lightning_modules.pl_simple_baseline import FlareLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [20]:
L.seed_everything(42, workers=True)

Seed set to 42


42

## Intialize Lightning module

Now we properly initalize the Lightning module to enable training, including passing the dictionary of metrics

In [21]:
metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

lit_model = FlareLightningModule(model, metrics, lr=cfg.learning_rate, batch_size=batch_size)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [22]:
project_name = cfg.wandb_project
run_name = "adi_vs_surya_fine_tuning"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).
- The **precision**, which now comes from `training.precision` in the config rather than being hardcoded.

### Mixed precision is a memory lever *and* a speed lever, and the right dtype depends on your GPU

The usual advice is "use bf16", and on an A100 or H100 that is right. On the T4 this workshop runs on, it is wrong. Ask the hardware:

```python
torch.cuda.get_device_capability()                       # (7, 5) -- Turing
torch.cuda.is_bf16_supported()                           # True   -- but read the next line
torch.cuda.is_bf16_supported(including_emulation=False)  # False  -- it is EMULATED
```

Turing's selling point is native **fp16** tensor cores; it has no bf16 units, so PyTorch emulates bf16. Timing one GEMM at this model's MLP shape (16384×1280×5120) on this GPU:

| dtype | time |
|---|---|
| `float16` | **9.3 ms** |
| `float32` | 56.8 ms |
| `bfloat16` | 90.1 ms |

So `bf16-mixed` here buys the memory saving and hands back a **60% slowdown versus plain fp32**. `16-mixed` gives identical memory at roughly 6× the speed. On SM80+ the ranking flips and `bf16-mixed` is the better choice — which is exactly why this is a config value and not a constant.

### Why `-mixed` and not `-true`

`bf16-true` and `16-true` cast the *weights*, which sounds strictly better for memory. Avoid both. Lightning casts the module in `Strategy.setup` **before** it builds the optimizer, so Adam's `exp_avg_sq` is allocated in bf16 — and with `beta2=0.999` each update is about 1/1000 of the running value, below bf16's ~1/256 resolution. The second moment stops accumulating. Measured at `lr=1e-4` over 20 steps, a parameter of magnitude 1.0 **does not move at all**, while the loss curve still looks alive.

`-mixed` keeps an fp32 master copy of everything trainable, which is what makes the optimizer's arithmetic trustworthy. The frozen 99.8% of the model does not need that protection, which is the trick the next few cells use.

In [23]:
# max_epochs deliberately overrides cfg.max_epochs (20) to keep this walkthrough short.
max_epochs = 2

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    # From the config now, not hardcoded: the right dtype is hardware dependent (see the
    # markdown above), and it is the single largest memory lever for this app's 2D head.
    precision=cfg.precision,
    accumulate_grad_batches=cfg.accumulate_grad_batches,
    # The sanity check runs under no_grad, so its own peak is small -- but it grows the
    # caching allocator's pool with a differently-shaped set of blocks right before
    # training's first allocation, and it spends two full 832 MiB host-to-device copies out
    # of a 10-sample dataset. Turn it back on once memory is no longer tight.
    num_sanity_val_steps=0,
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Fit the model

Finally we fit the model.  We pass the Lighting module, and our dataloaders.

In [24]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


wandb: Currently logged in as: adithyabhattsringeri (surya-ws2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


/home/jovyan/envs/surya_ws/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type      | Params | Mode  | FLOPs
----------------------------------------------------
0 | model | PeftModel | 358 M  | train | 0    
----------------------------------------------------
706 K     Trainable params
357 M     Non-trainable params
358 M     Total params
1,434.807 Total estimated model params size (MB)
537       Modules in train mode
0         Modules in eval mode
0         Total Flops


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


[W918 04:24:42.493747959 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:42.493921576 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:42.494008670 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:42.494019941 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())


[W918 04:24:43.744806510 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:43.744832395 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:43.744964609 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W918 04:24:43.745834438 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())


## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and DS into an end-2-end training loop and we are ready to experiment!